# Week 16: Advanced MLOps & Distributed Deep Learning
## Notebook 03: ZeRO (Zero Redundancy Optimizer) & Memory Breakdown

### 1. Conceptual Intuition & Real-World Use Case
When training large language models with mixed precision (FP16/BF16), most engineers assume the majority of GPU VRAM is consumed by model weights. **This is completely false.**

In reality, standard Adam optimizer states consume **4x more memory than the model weights themselves**!
Microsoft's **ZeRO (Zero Redundancy Optimizer)** (implemented in DeepSpeed and PyTorch FSDP) eliminates memory redundancy across data-parallel GPUs by sharding optimizer states, gradients, and model parameters.

---

### 2. Mathematical Foundations: Where Does VRAM Go?
For a model with $\Phi$ parameters trained in mixed precision (16-bit):

1. **Model Weights:** $2\Phi$ bytes (16-bit float).
2. **Gradients:** $2\Phi$ bytes (16-bit float).
3. **Adam Optimizer States:**
   - FP32 Master Weights (for numerical stability): $4\Phi$ bytes.
   - FP32 Momentum (First Moment): $4\Phi$ bytes.
   - FP32 Variance (Second Moment): $4\Phi$ bytes.
   - **Total Optimizer State:** $12\Phi$ bytes!

**Baseline Total Static Memory:**
$$2\Phi + 2\Phi + 12\Phi = 16\Phi \text{ bytes per parameter!}$$
*Example:* A 70B parameter model requires $70 \times 10^9 \times 16 = 1,120 \text{ GB}$ of static VRAM (14x 80GB A100 GPUs just to load parameters before activations!).

---

### 3. ZeRO Stages Explained:
Across $N_d$ data-parallel GPUs:
- **ZeRO Stage 1 ($P_{os}$):** Shards the optimizer states. Memory drops from $16\Phi \to 4\Phi + \frac{12\Phi}{N_d}$.
- **ZeRO Stage 2 ($P_{os+g}$):** Shards optimizer states AND gradients. Memory drops to $2\Phi + \frac{14\Phi}{N_d}$.
- **ZeRO Stage 3 ($P_{os+g+p}$):** Shards optimizer states, gradients, AND model parameters! Memory scales to:
  $$\frac{16\Phi}{N_d}$$
  Every parameter is gathered on-the-fly via AllGather during forward/backward passes and immediately freed!

In [ ]:
def calculate_training_memory(params_billions, num_gpus):
    phi = params_billions * 1e9  # Number of parameters
    
    # Static components in bytes
    weights_fp16 = 2 * phi
    grads_fp16 = 2 * phi
    optimizer_adam = 12 * phi
    total_baseline = weights_fp16 + grads_fp16 + optimizer_adam
    
    # ZeRO Stages (bytes per GPU)
    zero_1 = (weights_fp16 + grads_fp16) + (optimizer_adam / num_gpus)
    zero_2 = weights_fp16 + ((grads_fp16 + optimizer_adam) / num_gpus)
    zero_3 = total_baseline / num_gpus
    
    def to_gb(b):
        return b / (1024 ** 3)
        
    return {
        "params": f"{params_billions}B",
        "gpus": num_gpus,
        "baseline_gb": to_gb(total_baseline),
        "zero1_per_gpu_gb": to_gb(zero_1),
        "zero2_per_gpu_gb": to_gb(zero_2),
        "zero3_per_gpu_gb": to_gb(zero_3)
    }

models = [7, 13, 70]
gpu_counts = [8, 64]

print(f"{'Model':<8} {'GPUs':<6} {'Baseline Static':<16} {'ZeRO-1 / GPU':<14} {'ZeRO-2 / GPU':<14} {'ZeRO-3 / GPU':<14}")
print("-" * 75)

for m in models:
    for g in gpu_counts:
        stats = calculate_training_memory(m, g)
        print(f"{stats['params']:<8} {stats['gpus']:<6} {stats['baseline_gb']:>10.1f} GB   {stats['zero1_per_gpu_gb']:>10.1f} GB  {stats['zero2_per_gpu_gb']:>10.1f} GB  {stats['zero3_per_gpu_gb']:>10.1f} GB")

### 3. Step-by-Step Code Walkthrough
1. **Memory Accounting:** The calculation shows a 70B parameter model requires 1,043 GB baseline VRAM.
2. **ZeRO-1 on 64 GPUs:** Sharding Adam states cuts per-GPU memory from 1,043 GB down to 42.6 GB, fitting on standard 80GB A100s.
3. **ZeRO-3 on 64 GPUs:** Cuts per-GPU memory to just 16.3 GB, leaving ample headroom for sequence activations ($L^2$ attention context).

---

### 4. Industry Applications & Trade-offs
- **PyTorch FSDP (Fully Sharded Data Parallel):** Facebook's native implementation of ZeRO-3.
- **Communication Overhead:** ZeRO-3 increases communication by 50% compared to standard DDP because parameters must be AllGathered in every forward and backward step.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Adam require 12 bytes per parameter during FP16 training?**
> **Answer:** FP16 suffers from numerical underflow with small gradient updates. Adam maintains a master copy of weights in FP32 (4 bytes), first moment / momentum in FP32 (4 bytes), and second moment / variance in FP32 (4 bytes). Total = $4 + 4 + 4 = 12$ bytes per parameter.

**Q2: What is the difference between ZeRO-Offload and ZeRO-3?**
> **Answer:** ZeRO-3 shards states across GPU memories. ZeRO-Offload moves optimizer states and computation to host system CPU RAM and PCIe channels, allowing 13B models to train on a single consumer GPU with 24GB VRAM at the cost of PCIe transfer latency.